# ML Teaching Essentials — Model Diagnostics
## TL;DR — Plain English Introduction

**What separates good ML engineers from great ones?** Knowing WHY a model fails and HOW to fix it. This notebook teaches the diagnostic skills that interviewers test.

**The 5 most common ML failures (and fixes):**
1. **High bias (underfitting)** — model too simple → use more complex model or add features
2. **High variance (overfitting)** — model memorizes training data → add regularization, more data, dropout
3. **Vanishing gradients** — gradients near 0 in deep networks → use ReLU, batch norm, residual connections
4. **Exploding gradients** — gradients blow up → gradient clipping (`clip_grad_norm_`)
5. **Data leakage** — test data contaminated training → strict train/val/test splits before ANY processing

**No ML background?**
- Training loss = how wrong is my model on data it has seen?
- Validation loss = how wrong on data it has NOT seen?
- If train loss << val loss: overfitting (memorizing, not learning)
- If both losses are high: underfitting (not learning at all)

**Learning path:** 00/02 (ML fundamentals) → 00/06 (PyTorch) → This notebook → 05/01 (Deep learning) → 10/01 (Fine-tuning)

## Model Diagnostics — Concepts for Beginners

| Term | Plain English |
|------|---------------|
| **bias** | Systematic error — a biased model makes the same type of mistake consistently |
| **variance** | Sensitivity to training data — high-variance model changes dramatically with different training sets |
| **bias-variance tradeoff** | Simple models have high bias (underfit); complex models have high variance (overfit) |
| **underfitting** | Model is too simple to capture the pattern — both training and validation error are high |
| **overfitting** | Model memorises training data — low training error but high validation error |
| **learning curve** | Plot of train/val error vs training set size — reveals whether you need more data or less complexity |
| **validation curve** | Plot of train/val error vs hyperparameter — shows the underfitting->overfitting transition |
| **calibration** | A model is calibrated if its confidence scores match actual frequencies (70% prediction -> 70% correct) |
| **reliability diagram** | Plot of mean predicted probability vs actual fraction of positives — checks calibration |
| **Platt scaling** | Fits a logistic regression on top of raw model outputs to improve calibration |
| **temperature scaling** | Divides logits by a learnable temperature T before softmax — simplest calibration method |
| **early stopping** | Stop training when validation loss stops improving — a form of implicit regularisation |

## Beginner Teaching Frame

**Who should fully work through this notebook:** anyone who can train a model but does not yet trust their diagnostic judgment.

**How to study it on a first pass:**
- pause after each plot and say what failure pattern it reveals
- connect every visualization to a training decision
- treat this notebook as a lens for reading later notebooks more intelligently

**Common traps:** optimizing metrics blindly, ignoring validation behavior, and confusing a good-looking training curve with a good model.


## Canonical Resource Upgrade

Strong companion references:
- [scikit-learn model evaluation docs](https://scikit-learn.org/stable/model_selection.html)
- [Dive into Deep Learning](https://d2l.ai/)
- [Made With ML](https://madewithml.com/) for practical debugging habits


# ML Teaching Essentials — Model Diagnostics & Training

## What This Notebook Covers

These are the concepts that separate someone who *uses* ML from someone who *understands* it. Every professional ML interview will probe at least one of these. They were missing from the other notebooks — this one covers them all, with visualizations and biological examples.

## Learning Objectives
- [ ] Draw and interpret learning curves — diagnose overfitting vs underfitting
- [ ] Demonstrate the bias-variance tradeoff with actual data
- [ ] Compare L1 vs L2 vs no regularization on a real dataset
- [ ] Build early stopping from scratch and with callbacks
- [ ] Implement learning rate scheduling (step decay, cosine annealing)
- [ ] Plot multi-class confusion matrix and ROC curves
- [ ] Compute SHAP values and interpret individual predictions
- [ ] Debug 5 common training failures (NaN loss, stuck training, oscillating loss, etc.)
- [ ] Run a hyperparameter sensitivity analysis with plots

## Prerequisites
- ML Fundamentals (Notebook 00/02)
- NumPy + Matplotlib basics

## 🗺️ Prerequisites & Learning Path

| | |
|--|--|
| 🔑 Prerequisites | 00/02 (NumPy/matplotlib), 05/01 (deep learning training fundamentals) |
| 📍 You Are Here | Module 09/01 — Model Diagnostics & Training |
| ➡️ Next Steps | Any module requiring model training — this is a standalone reference |
| 🏁 Goal | Diagnose underfitting/overfitting from learning curves; compare regularization strategies; implement gradient flow monitoring |

### 🆕 From Scratch? Start Here:
1. [StatQuest bias-variance video](https://www.youtube.com/watch?v=EuBBz3bI-aA) — 10 min, best visual explanation
2. [sklearn learning_curve docs](https://scikit-learn.org/stable/modules/learning_curve.html) — hands-on starting point
3. 00/02 (this repo) — NumPy/matplotlib for plotting diagnostics
4. This notebook — model diagnostics
**Time:** 8–12h | **Difficulty:** ⭐⭐⭐

### 🔗 Cross-References
- Builds on: 00/02 (plotting), 05/01 (training loop patterns), 04/01 (ML fundamentals — bias-variance introduced there)
- Used by: ALL modules with trained models — apply these diagnostics to every notebook's training cell
- Standalone: This notebook can be read independently as a reference guide

In [ ]:
# Prerequisites cell — imports and setup
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression, make_classification
from sklearn.model_selection import train_test_split, learning_curve, validation_curve
from sklearn.linear_model import Ridge, Lasso, LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve
import torch
import torch.nn as nn
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print("All imports successful!")
print("Module 09: Model Diagnostics — bias-variance, regularization, gradient flow, SHAP")

> 📋 **Expected output:** A plot with two curves — training error (blue) and validation error (orange). If they converge at HIGH error → **high bias** (underfit). If large gap → **high variance** (overfit).
> ✅ Both curves visible and training error ≤ validation error.

## 🔧 Common Errors & Troubleshooting

| Error | Cause | Fix |
|-------|-------|-----|
| `ValueError: Input contains NaN` in sklearn | Missing values in feature matrix | Impute with `SimpleImputer` or drop rows with `dropna()` before fitting |
| Learning curve shows train and val both high loss | Underfitting (high bias) | Increase model complexity, add features, reduce regularization |
| Learning curve shows train low, val high loss | Overfitting (high variance) | More data, increase regularization, reduce model complexity, use dropout |
| Calibration curve shows overconfident predictions | Model outputs are not well-calibrated | Apply Platt scaling (`CalibratedClassifierCV`) or temperature scaling |
| `ConvergenceWarning` from sklearn | Optimizer didn't converge | Increase `max_iter`, scale features with `StandardScaler`, reduce learning rate |

## 🎤 Interview Questions

**Q1 (Easy):** What is the bias-variance tradeoff?
<details><summary>Answer</summary>
Bias is error from wrong assumptions (underfitting) — the model is too simple to capture the true pattern. Variance is error from sensitivity to training data fluctuations (overfitting) — the model memorises noise. Total error = Bias² + Variance + Irreducible noise. Simple models have high bias, low variance. Complex models have low bias, high variance. The goal is to find the sweet spot that minimises total error, typically through cross-validation.
</details>

**Q2 (Easy):** How do you read a learning curve, and what does it tell you about your model?
<details><summary>Answer</summary>
Plot training and validation error vs. training set size. If both curves are high and converged: underfitting (more data won't help, need a more complex model). If training error is low but validation error is high: overfitting (more data will help, or reduce complexity). If both curves converge to a low value: good fit. The gap between curves indicates variance; the final level indicates bias.
</details>

**Q3 (Medium):** What is model calibration and why does it matter for medical/biological predictions?
<details><summary>Answer</summary>
A calibrated model's predicted probability matches the true frequency — if it says "70% probability of disease," then 70% of such predictions should actually be positive. Many models (random forests, SVMs, neural nets) are poorly calibrated out of the box. In medicine/biology, miscalibrated probabilities lead to wrong clinical decisions. Fix with Platt scaling (logistic regression on outputs), isotonic regression, or temperature scaling (for neural nets). Evaluate with reliability diagrams and Expected Calibration Error (ECE).
</details>

**Q4 (Medium):** You have a model with 95% accuracy but your stakeholder is unhappy. What metrics should you look at instead?
<details><summary>Answer</summary>
Check: (1) Class distribution — if 95% of samples are negative, the model could just predict "negative" always. (2) Precision and recall per class — accuracy hides poor performance on the minority class. (3) F1-score or MCC for imbalanced data. (4) ROC-AUC and PR-AUC (Precision-Recall AUC is better for imbalanced data). (5) Confusion matrix to see exactly where errors occur. (6) Business-relevant metrics — e.g., cost of false negatives vs. false positives. Always report multiple metrics and choose based on the cost structure of the problem.
</details>

**Q5 (Hard):** Design a diagnostic workflow for a neural network that plateaus at mediocre validation performance. What would you check, in what order?
<details><summary>Answer</summary>
Systematic workflow: (1) Data: Check for label noise, class imbalance, data leakage, insufficient preprocessing. (2) Learning curves: Plot train/val loss over epochs — is it underfitting (both high) or overfitting (gap)? (3) Gradient diagnostics: Check for vanishing/exploding gradients (plot gradient norms per layer). (4) Architecture: Is the model capacity sufficient? Try larger model to see if training loss can go lower. (5) Hyperparameters: Learning rate (try LR finder), batch size, weight decay. (6) Features: Are inputs properly normalised? Feature importance analysis. (7) Regularization: If overfitting, add dropout, weight decay, data augmentation. (8) Optimiser: Switch Adam → SGD+momentum or vice versa. (9) Ensemble: Combine multiple models. Log everything in a systematic table; change one thing at a time.
</details>

## ✅ Notebook Complete

**You can now:**
- Diagnose underfitting and overfitting using learning curves
- Evaluate model calibration and apply correction methods
- Choose appropriate metrics beyond accuracy for imbalanced problems
- Apply a systematic debugging workflow for underperforming models

**Next:** → `10_openfold3_finetuning/00_openfold3_walkthrough.ipynb`